# For those interested: Training a Character-Level Language Model

This script implements a character-level language model and derieves word embeddings from it as described in the paper by **Akbik, Blythe & Vollgraf**

---

## 0. Setup

In [1]:
# Uncomment to install missing packages
# !pip install torch datasets seqeval scikit-learn matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
import numpy as np
import random, math, os, re
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device : cpu
PyTorch: 2.11.0


## 1. Load CoNLL-2003

The cell below tries three routes in order:
1. **Local files** – point `CONLL_DIR` to a folder containing `eng.train`, `eng.testa`, `eng.testb`
2. **HuggingFace `datasets`** – `load_dataset('conll2003')` (requires internet + dataset agreement)
3. **Built-in sample** – 50 real-looking CoNLL-style sentences so every downstream cell runs

The data format produced is a list of sentences, where each sentence is a list of `(token, BIO-tag)` tuples.

In [2]:
def hf_to_sentences(hf_split, id2label: Dict[int, str]) -> List[List[Tuple[str, str]]]:
    """Convert a HuggingFace conll2003 split to (token, tag) lists."""
    sentences = []
    for ex in hf_split:
        sent = [(tok, id2label[tag]) for tok, tag in zip(ex['tokens'], ex['ner_tags'])]
        sentences.append(sent)
    return sentences

raw_datasets = load_dataset("eriktks/conll2003",revision="convert/parquet")
labels = raw_datasets["train"]["ner_tags"].features.feature.names
id2label = {i: l for i, l in enumerate(labels)}
train_sentences = hf_to_sentences(raw_datasets['train'],      id2label)
dev_sentences   = hf_to_sentences(raw_datasets['validation'], id2label)
test_sentences  = hf_to_sentences(raw_datasets['test'],       id2label)

## 2. Vocabulary

In [3]:
class Vocabulary:
    def __init__(self, specials=None):
        self.item2idx: Dict[str, int] = {}
        self.idx2item: List[str] = []
        if specials:
            for s in specials: self.add(s)

    def add(self, item: str) -> int:
        if item not in self.item2idx:
            self.item2idx[item] = len(self.idx2item)
            self.idx2item.append(item)
        return self.item2idx[item]

    def __getitem__(self, item):
        return self.item2idx.get(item, self.item2idx.get('<unk>', 0))

    def __len__(self):
        return len(self.idx2item)


ALL_SENTS = train_sentences + dev_sentences + test_sentences

char_vocab  = Vocabulary(specials=['<pad>', '<unk>', '<bos>', '<eos>'])
label_vocab = Vocabulary(specials=['<pad>'])

for sent in ALL_SENTS:
    for tok, tag in sent:
        for ch in tok: char_vocab.add(ch)
        label_vocab.add(tag)
        
char_vocab.add(' ')

PAD_IDX = char_vocab['<pad>']
BOS_IDX = char_vocab['<bos>']
EOS_IDX = char_vocab['<eos>']

print(f'Character vocabulary : {len(char_vocab)} unique characters')
print(f'NER label vocabulary : {len(label_vocab)} labels')
print(f'Labels: {label_vocab.idx2item}')

Character vocabulary : 90 unique characters
NER label vocabulary : 10 labels
Labels: ['<pad>', 'B-ORG', 'O', 'B-MISC', 'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'I-MISC', 'I-LOC']


## 3. Character Language Model (§2.1)

The paper trains a **bidirectional character-level LSTM** language model on a 1-billion word corpus,
with `hidden_dim=2048` for one week on one GPU.  
Here we train a smaller model (configurable) **directly on the CoNLL-2003 text** to keep
the notebook self-contained. The architecture is identical to the paper.

> *To replicate paper results*, replace the LM training corpus with the
> 1B-word benchmark and use `LM_HIDDEN_DIM = 2048`.

In [5]:
class CharLanguageModel(nn.Module):
    """
    Character-level LSTM Language Model (paper §2.1).

    Trained to predict the next character conditioned on all previous characters.
    Hidden states encode syntactic-semantic information at each character position.
    """

    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256, num_layers=1, dropout=0.25):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                  batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
        self.drop       = nn.Dropout(dropout)
        # Eq. (3): fully-connected softmax (no bias), following Graves 2013
        self.proj       = nn.Linear(hidden_dim, vocab_size, bias=False)

    def forward(self, x, state=None):
        """x: (B, T). Returns logits (B,T,V), state, hidden_seq (B,T,H)."""
        emb             = self.drop(self.embedding(x))
        hidden, state   = self.lstm(emb, state)
        logits          = self.proj(self.drop(hidden))
        return logits, state, hidden

    def init_state(self, batch_size):
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=DEVICE)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=DEVICE)
        return h, c


# ── LM dataset ─────────────────────────────────────────────────────────────
from torch.utils.data import Dataset

class CharLMDataset(Dataset):
    def __init__(self, sentences, reverse=False, seq_len=128):
        # Flatten all sentences into one long character stream
        lines = []
        for sent in sentences:
            line = ' '.join(tok for tok, _ in sent)
            lines.append(line[::-1] if reverse else line)
        text   = '\n'.join(lines)
        indices = [BOS_IDX] + [char_vocab[ch] for ch in text] + [EOS_IDX]
        self.data    = torch.tensor(indices, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return max(1, (len(self.data) - 1) // self.seq_len)

    def __getitem__(self, idx):
        s = idx * self.seq_len
        e = min(s + self.seq_len, len(self.data) - 1)
        return self.data[s:e], self.data[s+1:e+1]


def collate_lm(batch):
    xs, ys = zip(*batch)
    xs = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)
    ys = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=-100)
    return xs, ys


def train_lm(model, loader, epochs=30, lr=20.0, clip=0.25):
    model.to(DEVICE)
    opt = optim.SGD(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(ignore_index=-100)
    history = []
    for epoch in range(1, epochs + 1):
        model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits, _, _ = model(x)
            B, T, V = logits.shape
            loss = crit(logits.reshape(B*T, V), y.reshape(B*T))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            total += loss.item()
        avg  = total / len(loader)
        history.append(avg)
        if epoch % 10 == 0:
            print(f'  epoch {epoch:3d}  loss {avg:.4f}  ppl {math.exp(avg):.2f}')
    return history

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
# Paper: hidden_dim=2048, trained 1 week on 1-billion word corpus.
# Notebook default: smaller model trained on CoNLL text for speed.
LM_EMBED   = 64
LM_HIDDEN  = 256   # set to 2048 + large corpus to replicate paper
LM_EPOCHS  = 40
LM_BATCH   = 32
LM_SEQLEN  = 128
LM_LR      = 20.0  # paper SGD lr

fwd_ds  = CharLMDataset(train_sentences, reverse=False, seq_len=LM_SEQLEN)
bwd_ds  = CharLMDataset(train_sentences, reverse=True,  seq_len=LM_SEQLEN)
fwd_ldr = DataLoader(fwd_ds, batch_size=LM_BATCH, shuffle=True, collate_fn=collate_lm)
bwd_ldr = DataLoader(bwd_ds, batch_size=LM_BATCH, shuffle=True, collate_fn=collate_lm)

fwd_lm = CharLanguageModel(len(char_vocab), LM_EMBED, LM_HIDDEN)
bwd_lm = CharLanguageModel(len(char_vocab), LM_EMBED, LM_HIDDEN)

print(f'LM corpus: {len(fwd_ds.data):,} chars  |  {len(fwd_ldr)} batches/epoch')
print('\nTraining FORWARD LM...')
fwd_hist = train_lm(fwd_lm, fwd_ldr, LM_EPOCHS, LM_LR)
print('\nTraining BACKWARD LM...')
bwd_hist = train_lm(bwd_lm, bwd_ldr, LM_EPOCHS, LM_LR)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for ax, hist, title in [(a1, fwd_hist, 'Forward LM'), (a2, bwd_hist, 'Backward LM')]:
    ax.plot(hist, color='steelblue')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('CE Loss'); ax.grid(alpha=.3)
plt.suptitle('Character LM Training (CoNLL-2003 text)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Contextual String Embedding Extraction (§2.2, Eq. 8)

$$w_i^{\text{CharLM}} = \left[\, h^f_{t_{i+1}-1}\; ;\; h^b_{t_i - 1} \,\right]$$

- Forward LM hidden state **after the last character** of word $i$  
- Backward LM hidden state **before the first character** of word $i$  
- These are concatenated to form the final word embedding

In [ ]:
@torch.no_grad()
def extract_contextual_embeddings(
    fwd_model: CharLanguageModel,
    bwd_model: CharLanguageModel,
    sentence:  List[Tuple[str, str]],
) -> torch.Tensor:
    """
    Extract contextual string embeddings for all words in a sentence.

    Returns: (num_words, 2 * hidden_dim)
    """
    fwd_model.eval(); bwd_model.eval()
    words  = [tok for tok, _ in sentence]
    joined = ' '.join(words)
    L      = len(joined)

    # Encode with BOS prefix
    fwd_ids = [BOS_IDX] + [char_vocab[ch] for ch in joined]
    bwd_ids = [BOS_IDX] + [char_vocab[ch] for ch in joined[::-1]]

    # Locate word character boundaries in `joined`
    boundaries, pos = [], 0
    for w in words:
        boundaries.append((pos, pos + len(w) - 1))  # (t_start, t_end), inclusive
        pos += len(w) + 1  # +1 for space separator

    # Forward pass  → hidden states indexed 0..L (0 = after BOS)
    fwd_t = torch.tensor(fwd_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    _, _, fwd_h = fwd_model(fwd_t)   # (1, L+1, H)
    fwd_h = fwd_h.squeeze(0)         # (L+1, H)

    # Backward pass (reversed sentence) → hidden states
    bwd_t = torch.tensor(bwd_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    _, _, bwd_h = bwd_model(bwd_t)   # (1, L+1, H)
    bwd_h = bwd_h.squeeze(0)         # (L+1, H)

    embeddings = []
    for t_start, t_end in boundaries:
        # fwd: state AFTER last char of word (t_end+1 because of BOS offset)
        h_f = fwd_h[min(t_end + 1, fwd_h.size(0) - 1)]
        # bwd: state equivalent to BEFORE first char of word
        # In the reversed string, original index t_start → reversed index L-1-t_start
        # +1 for BOS offset → index (L - t_start) in bwd_h
        h_b = bwd_h[min(L - t_start, bwd_h.size(0) - 1)]
        embeddings.append(torch.cat([h_f, h_b], dim=0))

    return torch.stack(embeddings, dim=0)   # (n_words, 2H)


# ── Sanity check ───────────────────────────────────────────────────────────
ex = train_sentences[0]
emb = extract_contextual_embeddings(fwd_lm, bwd_lm, ex)
print('Sentence:', [w for w, _ in ex])
print(f'Embedding shape: {list(emb.shape)}  (expected [{len(ex)}, {2*LM_HIDDEN}])')